In [1]:
import tensorflow as tf
import tensorflow.keras as keras
import pickle

In [2]:
from tensorflow.compat.v1 import ConfigProto
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.8
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

KeyboardInterrupt: 

In [ ]:
num_classes = 3
initial_lr = 0.1
weight_decay = 1e-4
epochs = 200
warmup_epochs = 5
batch_size = 128
image_size = 32

In [ ]:
with open('data.pkl', 'rb') as f:
    [x_train,y_train,x_test,y_test]=pickle.load(f)
y_train_onehot=tf.keras.utils.to_categorical(y_train,num_classes=4)
y_test_onehot=tf.keras.utils.to_categorical(y_test,num_classes=4)

In [ ]:
from network import build_resnet18
from train import WarmUpCosine, CustomWeightDecaySGD, LastNSaver, make_train_ds, make_test_ds

In [ ]:
NN_18=[32,32,32,64,64,128,128,256,256]

In [ ]:
model = build_resnet18(NN_18,num_classes=4)

In [ ]:
total_steps = epochs * (x_train.shape[0] // batch_size)
warmup_steps = warmup_epochs * (x_train.shape[0] // batch_size)
lr_schedule = WarmUpCosine(initial_lr, total_steps, warmup_steps)
optimizer = CustomWeightDecaySGD(
    weight_decay=weight_decay,
    learning_rate=lr_schedule,
    momentum=0.9,
    nesterov=True
)

In [ ]:
loss_fn=tf.keras.losses.CategoricalCrossentropy()
model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=['accuracy']
)

In [ ]:
saver = LastNSaver(n=20)

In [ ]:
train_ds = make_train_ds(
    x_train, y_train_onehot,
    batch_size=batch_size,
    image_size=32, pad=4,
    mixup_alpha=0.2   # 0.2~0.4 都常用；过拟合明显就加大
)
test_ds = make_test_ds(x_test, y_test_onehot, batch_size=batch_size)

model.fit(
    train_ds,
    epochs=epochs,
    validation_data=test_ds,
    verbose=2,
    callbacks=[saver]
)

In [ ]:
model.save("Res_18.h5")